# 10 · Preparing any video: several people, with hair, in any pose

The preprocessing stage for videos that are not turntables: people with hair, moving freely,
more than one in the frame. Every person in every frame gets:

| What | Model | Confidence shown |
| --- | --- | --- |
| Who is where: an outline per person, kept through the video | RT-DETR finds them, SAM 2.1 follows each | the detection's score |
| 308 keypoints: body, feet, face, hands | Sapiens2 pose, on the person alone | per keypoint |
| 21 landmarks per hand | MediaPipe, in a crop around each wrist | per hand |
| 478 face landmarks | MediaPipe, in a crop around the face | the face keypoints' |
| 28 body parts: hair, face and neck, torso, upper and lower arms and legs, hands, feet, lips, teeth, tongue, clothes | Sapiens2, on a crop of the person | per pixel |
| Face parts: brows, eyes, irises, nose, lips, mouth opening | filled outlines of the face landmarks | the face's |

One person is followed: you pick them on one frame (several can be in the video; the others
are not followed), and from then on every model sees that person alone: everything outside their
outline is greyed out first, so no model can take a limb from someone touching them.

SAM answers *who*: it keeps the person's identity, also when someone passes in front.
Sapiens2 answers *what*: it labels every pixel of a person but cannot tell people apart, so it
sees one person at a time, cropped around their outline, and its labels are kept only inside
that person's outline. The face parts are not guessed by a model: MediaPipe's landmarks have
fixed meanings (the eyelids, the lips' inner and outer edge), so their outlines are exact even
where the face is a few dozen pixels.

Every frame is drawn into an overlay video; `show_frame(i)` draws one frame large; the last cells
show each body part's coverage and chance of reconstruction per person, every confidence over
time, and where the time went. Results go to `outputs/prepare/<name>/`.

In [ ]:
import time
from contextlib import contextmanager
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Video, display
from mediapipe.tasks.python import BaseOptions, vision
from transformers import (
    AutoImageProcessor,
    Sapiens2ForPoseEstimation,
    Sapiens2ForSemanticSegmentation,
)

from avatar_splats.cameras.turntable import body_yaw
from avatar_splats.config import load_config
from avatar_splats.data.video import probe, read_frames, write_video
from avatar_splats.preprocess.detection import detect_people
from avatar_splats.preprocess.face import download_face_landmarker
from avatar_splats.preprocess.hands import detect_hands, download_hand_landmarker
from avatar_splats.preprocess.keypoints import COCO_KEYPOINTS
from avatar_splats.preprocess.segmentation import mask_boxes, track_people_scored
from avatar_splats.preprocess.triage import rank_shots

CONFIG = "../configs/default.yaml"
config = load_config(CONFIG)
models = config.preprocess
device = models.device

## Settings

The stretch to prepare comes from notebook 11's triage of the video (`TRIAGE`): shot `SHOT` of
its ranking (0 the best), `MAX_FRAMES` sampled frames (every `FRAME_STEP`-th) around that shot's
best sample, cut at the shot's ends. With `TRIAGE = None` it starts at frame `START` instead.
SAM keeps all frames in memory while it tracks, so keep `MAX_FRAMES` moderate; frames larger than `MAX_SIDE` are scaled
down first, and every pixel coordinate below is in the scaled frames.

In [ ]:
VIDEO = Path("../examples/live-performance/6.mp4")  # relative to notebooks/
NAME = VIDEO.stem
TRIAGE = config.outputs / "triage" / NAME / "triage.npz"  # None: start at START
SHOT = 0  # the triage's shots, best first
START = 0  # the first frame of the video to prepare, without TRIAGE
FRAME_STEP = 1  # 5: every fifth frame (SAM then follows bigger moves between frames)
MAX_FRAMES = 120
MAX_SIDE = 1280  # pixels: longer sides are scaled down to this

SELECT_FRAME = None  # the frame to pick the person on; None: the triage's best sample (or,
# without TRIAGE, the one showing the most people)
DETECT_EVERY = 5  # frames searched for that frame without TRIAGE or SELECT_FRAME
MIN_DETECTION = 0.5  # the detector's score to accept a person
# SAM 2.1 in four sizes: hiera-tiny, -small, -base-plus, -large (the pipeline's, the slowest)
SEGMENTER = models.segmenter
# A person's frame only goes on to the other models when SAM is sure of it:
MIN_PRESENCE = 0.5  # SAM's probability that the person shows at all
MIN_MASK_CERTAINTY = 0.8  # the mean probability SAM gives the pixels of its mask
MIN_AREA = 0.002  # the mask's share of the frame: smaller people are too small to read
ISOLATE_MARGIN = 0.02  # the outline is widened by this share of the person's size before
# everything outside it is greyed out (so fingertips and hair ends at its edge survive)
POSE_MODEL = "facebook/sapiens2-pose-1b"  # also -0.4b, -0.8b, -5b
POSE_BATCH = 4  # frames per batch through the pose model
PARTS_MODEL = "facebook/sapiens2-seg-1b"  # also -0.4b (faster, coarser) and -5b
PARTS_BATCH = 8  # person crops per batch through Sapiens2
PERSON_MARGIN = 0.08  # crop margin around a person's outline, in shares of its larger side
MIN_KEYPOINT = 0.3  # face keypoints below this do not place the face crop
LANDMARK_CROP = 1.6  # face crop for the landmarks, in spans of the face keypoints
HEAD_CROP = 2.4  # crop shown in the head panel, in the same spans
PANEL_HEIGHT = 720  # pixels per panel in the overlay video

# How a body part's chance of being reconstructed well is scored (see "Seen in" below):
MIN_PART_PIXELS = 400  # a body part counts as seen in a frame from this many pixels of it
MIN_FEATURE_PIXELS = 20  # the same for a face part (an eye is small in a full-body frame)
YAW_BINS = 8  # directions around the person (45° each) a part should be seen from
ENOUGH_FRAMES = 20  # frames seen in that count as plenty
ENOUGH_PIXELS = 5000  # median pixels per sighting that count as plenty of detail (~70²)
ENOUGH_FEATURE_PIXELS = 400  # the same for a face part (~20²)

out_dir = config.outputs / "prepare" / NAME
out_dir.mkdir(parents=True, exist_ok=True)
timings = {}


@contextmanager
def timed(step):
    # adds the wall-clock seconds of the block to timings[step], GPU work included
    torch.cuda.synchronize()
    started = time.perf_counter()
    yield
    torch.cuda.synchronize()
    timings[step] = timings.get(step, 0.0) + time.perf_counter() - started

## Frames

In [ ]:
info = probe(VIDEO)
first, count, triage_frame = START, MAX_FRAMES, None  # triage_frame: the shot's best sample
if TRIAGE is not None:
    triage = np.load(TRIAGE)
    shot = rank_shots(triage["times"], triage["shot"], triage["score"], info.duration)[SHOT]
    begin, end = round(shot.begin * info.fps), round(shot.end * info.fps)  # end excluded
    best = round(triage["times"][shot.best] * info.fps)
    span = MAX_FRAMES * FRAME_STEP
    first = int(np.clip(best - span // 2, begin, max(begin, end - span)))
    count = min(MAX_FRAMES, -(-(end - first) // FRAME_STEP))
    triage_frame = min((best - first) // FRAME_STEP, count - 1)
    print(
        f"shot {SHOT} of {TRIAGE}: score {shot.score:.2f}, {shot.begin:.1f}-{shot.end:.1f} s, "
        f"best sample at {triage['times'][shot.best]:.1f} s"
    )
with timed("decode"):
    frames = list(read_frames(VIDEO, step=FRAME_STEP, max_frames=count, start=first))
    scale = min(1.0, MAX_SIDE / max(frames[0].shape[:2]))
    if scale < 1:
        size = (round(frames[0].shape[1] * scale), round(frames[0].shape[0] * scale))
        frames = [cv2.resize(frame, size, interpolation=cv2.INTER_AREA) for frame in frames]
HEIGHT, WIDTH = frames[0].shape[:2]
COUNT = len(frames)
frame_indices = first + np.arange(COUNT) * FRAME_STEP
fps = info.fps / FRAME_STEP
print(f"{VIDEO.name}: {info.width}x{info.height}, {info.fps:g} fps, {info.num_frames} frames")
print(f"prepared: {COUNT} frames ({frame_indices[0]}..{frame_indices[-1]}) at {WIDTH}x{HEIGHT}")

## Choose the person

The detector looks at `SELECT_FRAME`, or the triage's best sample of the shot, or (with neither)
searches every `DETECT_EVERY`-th frame for the one showing the most people, and numbers the people
it finds, the surest first. The suggested person is the triage's main person, the largest one (or,
without a triage, the surest); set `TARGET` in the next cell to another number to follow someone
else.

In [ ]:
with timed("detect people"):
    if SELECT_FRAME is not None:
        searched = [SELECT_FRAME]
    elif triage_frame is not None:
        searched = [triage_frame]
    else:
        searched = list(range(0, COUNT, DETECT_EVERY))
    found = detect_people([frames[i] for i in searched], models.detector, device, MIN_DETECTION)
    best = max(range(len(found)), key=lambda i: (len(found[i]), found[i][:, 4].sum()))
if not len(found[best]):
    raise LookupError("no person found; try another SELECT_FRAME")
select_frame, candidates = searched[best], found[best]
sizes = (candidates[:, 2] - candidates[:, 0]) * (candidates[:, 3] - candidates[:, 1])
suggested = int(sizes.argmax()) if TRIAGE is not None else 0
picture = frames[select_frame].copy()
for n, (x0, y0, x1, y1, score) in enumerate(candidates):
    colour = (255, 60, 60) if n == 0 else (60, 160, 255)
    cv2.rectangle(picture, (int(x0), int(y0)), (int(x1), int(y1)), colour, 3)
    cv2.putText(
        picture,
        f"{n}: {score:.2f}",
        (int(x0) + 6, int(y0) + 34),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.1,
        colour,
        3,
    )
plt.figure(figsize=(12, 12 * HEIGHT / WIDTH))
plt.imshow(picture)
plt.title(f"frame {frame_indices[select_frame]}: {len(candidates)} people, suggested {suggested}")
plt.axis("off")
plt.show()

## Follow them

SAM 2.1 follows the chosen person from that frame, forwards and backwards. It also judges every
mask: its probability that the person shows at all (*presence*), and the mean probability of the
mask's pixels (*certainty*). A frame goes on to the other models only if both clear their
thresholds and the mask is large enough; elsewhere the person counts as not there, so no model
spends time on an empty or doubtful outline.

In [ ]:
TARGET = suggested  # or the number of another person from the picture above

PEOPLE = 1  # the arrays below keep a person axis, of length 1
with timed("outline (SAM)"):
    tracked = track_people_scored(
        frames, select_frame, SEGMENTER, device, boxes=candidates[TARGET : TARGET + 1]
    )
masks, presence, mask_certainty = tracked.masks, tracked.presence, tracked.certainty  # (P, N, ...)
area = masks.reshape(PEOPLE, COUNT, -1).mean(-1)
usable = (presence >= MIN_PRESENCE) & (np.nan_to_num(mask_certainty) >= MIN_MASK_CERTAINTY)
usable &= area >= MIN_AREA
masks[~usable] = False  # the other models see only usable outlines
boxes = np.stack([mask_boxes(m) for m in masks])  # (P, N, 4), NaN where unusable
present = presence >= MIN_PRESENCE
unsure = present & (np.nan_to_num(mask_certainty) < MIN_MASK_CERTAINTY)
small = present & (area < MIN_AREA)
print(
    f"person {TARGET}: usable in {usable[0].sum()} of {COUNT} frames (absent "
    f"{(~present[0]).sum()}, unsure {unsure[0].sum()}, too small {small[0].sum()})"
)

GREY = np.array([124, 116, 104], np.uint8)  # the models' mean colour: reads as nothing


def isolated(p, i):
    # frame i with everything outside person p's (widened) outline greyed out
    mask = masks[p, i]
    if not mask.any():
        return np.broadcast_to(GREY, frames[i].shape).copy()
    x0, y0, x1, y1 = boxes[p, i]
    radius = max(1, round(ISOLATE_MARGIN * max(x1 - x0, y1 - y0)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    keep = cv2.dilate(mask.astype(np.uint8), kernel).astype(bool)
    return np.where(keep[..., None], frames[i], GREY)


plt.figure(figsize=(12, 12 * HEIGHT / WIDTH))
plt.imshow(isolated(0, select_frame))
plt.title("what the models see of them")
plt.axis("off")
plt.show()

## Keypoints (Sapiens2 pose) and hands

Sapiens2 pose sees the person alone, in a crop around their outline, and gives 308 points: the
first 17 are COCO's body keypoints in COCO's order (which the hands, the face crops and the yaw
use), then the feet, 68 face points and the hands. Every model from here on skips the person's
unusable frames: their box is NaN, their outline empty.

In [ ]:
pose_processor = AutoImageProcessor.from_pretrained(POSE_MODEL)
pose_model = Sapiens2ForPoseEstimation.from_pretrained(POSE_MODEL).to(device).eval()
all_keypoints = np.full((PEOPLE, COUNT, 308, 3), np.nan)
with timed("keypoints (Sapiens2 pose)"), torch.inference_mode():
    for p in range(PEOPLE):
        valid = np.flatnonzero(usable[p])
        for start in range(0, len(valid), POSE_BATCH):
            batch = valid[start : start + POSE_BATCH]
            images = [isolated(p, i) for i in batch]
            coco_boxes = [[[x0, y0, x1 - x0, y1 - y0]] for x0, y0, x1, y1 in boxes[p, batch]]
            inputs = pose_processor(images=images, boxes=coco_boxes, return_tensors="pt").to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = pose_model(**inputs)
            results = pose_processor.post_process_pose_estimation(outputs, boxes=coco_boxes)
            for i, (person,) in zip(batch, results, strict=True):
                all_keypoints[p, i, :, :2] = person["keypoints"].float().cpu().numpy()
                all_keypoints[p, i, :, 2] = person["scores"].float().cpu().numpy()
keypoints = all_keypoints[:, :, :17]  # COCO's 17, for the hands, the face crops and the yaw

hands = np.full((PEOPLE, COUNT, 2, 21, 3), np.nan)
hand_model = download_hand_landmarker(models.hand_landmarker)
for p in range(PEOPLE):
    with timed("hands"):
        hands[p] = detect_hands([isolated(p, i) for i in range(COUNT)], keypoints[p], hand_model)
    left, right = (np.isfinite(hands[p, :, s, 0, 0]).sum() for s in range(2))
    mean = np.nanmean(keypoints[p, ..., 2])
    print(f"person {p}: mean keypoint confidence {mean:.2f}, hands found L {left}, R {right}")

## Face landmarks, per person

Cropped around each person's face keypoints first: in a full-body frame a face is a few dozen
pixels. The landmarker gives no score of its own, so a face's confidence is the mean
confidence of the face keypoints that placed its crop.

In [ ]:
def face_box(points, span):
    # x0, y0, x1, y1 of a square around the confident face keypoints, or None
    face = points[:5]
    face = face[face[:, 2] >= MIN_KEYPOINT]
    if len(face) < 3:
        return None
    centre = face[:, :2].mean(axis=0)
    half = span * max(np.ptp(face[:, 0]), np.ptp(face[:, 1]), 8.0)
    x0, y0 = np.maximum(np.floor(centre - half), 0).astype(int)
    x1, y1 = np.minimum(np.ceil(centre + half), (WIDTH, HEIGHT)).astype(int)
    return (x0, y0, x1, y1) if x1 - x0 >= 16 and y1 - y0 >= 16 else None


face_landmarks = np.full((PEOPLE, COUNT, 478, 3), np.nan)
face_confidence = np.full((PEOPLE, COUNT), np.nan)
landmark_boxes = [
    [face_box(points, LANDMARK_CROP) for points in keypoints[p]] for p in range(PEOPLE)
]
head_boxes = [[face_box(points, HEAD_CROP) for points in keypoints[p]] for p in range(PEOPLE)]
model_path = download_face_landmarker(models.face_landmarker)
options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.IMAGE,
)
with timed("face landmarks"), vision.FaceLandmarker.create_from_options(options) as landmarker:
    for p in range(PEOPLE):
        for i, box in enumerate(landmark_boxes[p]):
            if box is None:
                continue
            x0, y0, x1, y1 = box
            crop = np.ascontiguousarray(isolated(p, i)[y0:y1, x0:x1])
            result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=crop))
            if result.face_landmarks:
                points = np.array([(q.x, q.y, q.z) for q in result.face_landmarks[0]])
                face_landmarks[p, i] = points * (x1 - x0, y1 - y0, x1 - x0) + (x0, y0, 0)
                face = keypoints[p, i, :5, 2]
                face_confidence[p, i] = face[face >= MIN_KEYPOINT].mean()
for p in range(PEOPLE):
    print(f"person {p}: face found in {np.isfinite(face_confidence[p]).sum()} of {COUNT} frames")

## Body parts: Sapiens2, one person at a time

Sapiens2 takes portrait images of 1024 × 768 and stretches whatever it gets to that size, so
each person is cropped around their outline (with a margin), widened to 3:4 and padded where
the crop leaves the frame. Its labels and their confidence (the likeliest class's probability)
are pasted back inside that person's outline only; elsewhere the label is 255 (not this
person).

In [ ]:
PART_NAMES = (
    "Background", "Apparel", "Eyeglass", "Face_Neck", "Hair", "Left_Foot", "Left_Hand",
    "Left_Lower_Arm", "Left_Lower_Leg", "Left_Shoe", "Left_Sock", "Left_Upper_Arm",
    "Left_Upper_Leg", "Lower_Clothing", "Right_Foot", "Right_Hand", "Right_Lower_Arm",
    "Right_Lower_Leg", "Right_Shoe", "Right_Sock", "Right_Upper_Arm", "Right_Upper_Leg", "Torso",
    "Upper_Clothing", "Lower_Lip", "Upper_Lip", "Lower_Teeth", "Upper_Teeth", "Tongue",
)  # fmt: skip
PART = {name: k for k, name in enumerate(PART_NAMES)}
INPUT_HEIGHT, INPUT_WIDTH = 1024, 768


def person_crop(box):
    # x0, y0, x1, y1 of a 3:4 crop around a (4,) box, possibly reaching outside the frame
    x0, y0, x1, y1 = box
    margin = PERSON_MARGIN * max(x1 - x0, y1 - y0)
    x0, y0, x1, y1 = x0 - margin, y0 - margin, x1 + margin, y1 + margin
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    height = max(y1 - y0, (x1 - x0) * INPUT_HEIGHT / INPUT_WIDTH)
    width = height * INPUT_WIDTH / INPUT_HEIGHT
    return (
        round(cx - width / 2),
        round(cy - height / 2),
        round(cx + width / 2),
        round(cy + height / 2),
    )


def cut(frame, crop):
    # the crop's pixels, black where it leaves the frame
    x0, y0, x1, y1 = crop
    out = np.zeros((y1 - y0, x1 - x0, 3), np.uint8)
    fx0, fy0, fx1, fy1 = max(x0, 0), max(y0, 0), min(x1, WIDTH), min(y1, HEIGHT)
    if fx1 > fx0 and fy1 > fy0:
        out[fy0 - y0 : fy1 - y0, fx0 - x0 : fx1 - x0] = frame[fy0:fy1, fx0:fx1]
    return out


processor = AutoImageProcessor.from_pretrained(PARTS_MODEL)
parts_model = Sapiens2ForSemanticSegmentation.from_pretrained(PARTS_MODEL).to(device).eval()
body_parts = np.full((PEOPLE, COUNT, HEIGHT, WIDTH), 255, np.uint8)
part_confidence = np.zeros((PEOPLE, COUNT, HEIGHT, WIDTH), np.float16)
jobs = [
    (p, i, person_crop(boxes[p, i]))
    for p in range(PEOPLE)
    for i in range(COUNT)
    if masks[p, i].any()
]
with timed("body parts (Sapiens2)"), torch.inference_mode():
    for start in range(0, len(jobs), PARTS_BATCH):
        batch = jobs[start : start + PARTS_BATCH]
        images = [cut(isolated(p, i), crop) for p, i, crop in batch]
        inputs = processor(images=images, return_tensors="pt")
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = parts_model(**inputs.to(device)).logits.float()
        for (p, i, (x0, y0, x1, y1)), logit in zip(batch, logits, strict=True):
            size = (y1 - y0, x1 - x0)
            logit = F.interpolate(logit[None], size, mode="bilinear", align_corners=False)
            probability, label = logit.softmax(dim=1)[0].max(dim=0)
            fx0, fy0, fx1, fy1 = max(x0, 0), max(y0, 0), min(x1, WIDTH), min(y1, HEIGHT)
            inside = masks[p, i, fy0:fy1, fx0:fx1]
            window = (slice(fy0 - y0, fy1 - y0), slice(fx0 - x0, fx1 - x0))
            labels = label[window].byte().cpu().numpy()
            sure = probability[window].half().cpu().numpy()
            body_parts[p, i, fy0:fy1, fx0:fx1][inside] = labels[inside]
            part_confidence[p, i, fy0:fy1, fx0:fx1][inside] = sure[inside]
hair = body_parts == PART["Hair"]
for p in range(PEOPLE):
    share = hair[p].sum() / max(masks[p].sum(), 1)
    frames_with_hair = (hair[p].reshape(COUNT, -1).sum(1) > 0).sum()
    print(f"person {p}: hair in {frames_with_hair} frames, {share:.1%} of their pixels")

## Face parts from the face landmarks

MediaPipe's outlines of the eyes, irises and lips are closed loops of landmarks; the brows are
closed from their upper and lower rows; the nose is the hull of its landmarks. Filled in, they
give each face's parts exactly where the landmarks put them. Order: nose, brows, eyes, irises,
lips, then the mouth opening (inside the lips' inner edge) over them.

In [ ]:
FACE_PART_NAMES = ("none", "nose", "brows", "eyes", "irises", "lips", "mouth opening")
C = vision.FaceLandmarksConnections


def loops(connections):
    # the closed loops of landmark indices a set of edges forms
    neighbours = {}
    for edge in connections:
        neighbours.setdefault(edge.start, []).append(edge.end)
        neighbours.setdefault(edge.end, []).append(edge.start)
    unvisited, found = set(neighbours), []
    while unvisited:
        start = unvisited.pop()
        loop, previous, current = [start], None, start
        while True:
            following = [n for n in neighbours[current] if n != previous]
            if not following or following[0] == start:
                break
            previous, current = current, following[0]
            unvisited.discard(current)
            loop.append(current)
        found.append(loop)
    return found


LIPS = loops(C.FACE_LANDMARKS_LIPS)  # the outer and the inner edge
BROWS = (
    [70, 63, 105, 66, 107, 55, 65, 52, 53, 46],  # one brow: upper row, then lower row back
    [300, 293, 334, 296, 336, 285, 295, 282, 283, 276],
)
EYES = loops(C.FACE_LANDMARKS_LEFT_EYE) + loops(C.FACE_LANDMARKS_RIGHT_EYE)
IRISES = loops(C.FACE_LANDMARKS_LEFT_IRIS) + loops(C.FACE_LANDMARKS_RIGHT_IRIS)
NOSE = sorted({e.start for e in C.FACE_LANDMARKS_NOSE} | {e.end for e in C.FACE_LANDMARKS_NOSE})


def face_parts_of(points):
    # (H, W) uint8 face part labels from (478, 3) landmarks
    labels = np.zeros((HEIGHT, WIDTH), np.uint8)
    xy = points[:, :2].astype(np.float32)

    def fill(indices, value):
        cv2.fillPoly(labels, [np.round(xy[indices]).astype(np.int32)], value)

    hull = cv2.convexHull(xy[NOSE], returnPoints=False)[:, 0]
    fill([NOSE[k] for k in hull], 1)
    for brow in BROWS:
        fill(brow, 2)
    for eye in EYES:
        fill(eye, 3)
    for iris in IRISES:
        fill(iris, 4)
    outer, inner = sorted(LIPS, key=lambda loop: -cv2.contourArea(xy[loop]))
    fill(outer, 5)
    fill(inner, 6)
    return labels


face_parts = np.zeros((PEOPLE, COUNT, HEIGHT, WIDTH), np.uint8)
with timed("face parts (landmarks)"):
    for p in range(PEOPLE):
        for i in range(COUNT):
            if np.isfinite(face_landmarks[p, i]).all():
                face_parts[p, i] = face_parts_of(face_landmarks[p, i])
print("face part pixels per person:", [int((face_parts[p] > 0).sum()) for p in range(PEOPLE)])

## Save

In [ ]:
np.savez_compressed(
    out_dir / "prepared.npz",
    frame_indices=frame_indices,
    scale=scale,
    masks=masks,
    presence=presence,
    mask_certainty=mask_certainty,
    usable=usable,
    boxes=boxes,
    keypoints=all_keypoints,
    hands=hands,
    face_landmarks=face_landmarks,
    face_confidence=face_confidence,
    body_parts=body_parts,
    part_confidence=part_confidence,
    face_parts=face_parts,
    hair=hair,
    part_names=np.array(PART_NAMES),
    face_part_names=np.array(FACE_PART_NAMES),
)
print("written to", out_dir / "prepared.npz")

## Every frame

Four panels per frame:

1. **Detections**: each person's outline in their colour, body keypoints coloured by
   confidence (red 0 to green 1) with the skeleton, hands (yellow, their score by the wrist),
   face landmarks (cyan).
2. **Body parts** (Sapiens2), every person.
3. **Heads**, close up, one row per person: face parts from the landmarks and the hair's
   outline (magenta).
4. **Confidence** of the body parts per pixel; dark is unsure.

In [ ]:
COCO_BONES = ((0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
              (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16))  # fmt: skip
HAND_BONES = ((0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10),
              (10, 11), (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18),
              (18, 19), (19, 20), (0, 17))  # fmt: skip
PERSON_COLOURS = ((60, 220, 60), (255, 140, 0), (80, 160, 255), (230, 60, 230))
PART_COLOURS = np.concatenate(
    [plt.get_cmap("tab20")(np.arange(20)), plt.get_cmap("tab20b")(np.arange(9))]
)
PART_COLOURS = (PART_COLOURS[:, :3] * 255).astype(np.uint8)
PART_COLOURS[0] = 0
FACE_COLOURS = np.array(
    [[0, 0, 0], [230, 160, 60], [120, 70, 30], [240, 240, 240], [40, 120, 220], [220, 60, 90],
     [60, 0, 20]],
    np.uint8,
)  # fmt: skip


def confidence_colour(value):
    return (int(255 * (1 - value)), int(255 * value), 0)


def outline(image, mask, colour, thickness=2):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(image, contours, -1, colour, thickness)


def tint(image, labels, colours, alpha=0.6, skip=(0, 255)):
    mixed = image.astype(np.float32)
    region = ~np.isin(labels, skip)
    mixed[region] = (1 - alpha) * mixed[region] + alpha * colours[labels[region] % len(colours)]
    return mixed.astype(np.uint8)


def label(image, text, y=28, scale=0.8):
    for colour, width in (((0, 0, 0), 4), ((255, 255, 255), 2)):
        cv2.putText(
            image, text, (10, y), cv2.FONT_HERSHEY_SIMPLEX, scale, colour, width, cv2.LINE_AA
        )


def head_rows(i, height):
    # one close-up per person, stacked: face parts and the hair's outline
    side = height // PEOPLE
    rows = []
    for p in range(PEOPLE):
        box = head_boxes[p][i]
        if box is None:
            tile = np.full((side, side, 3), 40, np.uint8)
            label(tile, f"person {p}: no face", scale=0.6)
        else:
            x0, y0, x1, y1 = box
            crop = tint(frames[i][y0:y1, x0:x1], face_parts[p, i, y0:y1, x0:x1], FACE_COLOURS, 0.5)
            outline(crop, hair[p, i, y0:y1, x0:x1], (255, 0, 255), 1)
            tile = cv2.resize(crop, (side, side), interpolation=cv2.INTER_NEAREST)
            confidence = face_confidence[p, i]
            text = (
                f"person {p}: face {confidence:.2f}" if np.isfinite(confidence) else f"person {p}"
            )
            label(tile, text, scale=0.6)
        cv2.rectangle(tile, (0, 0), (side - 1, side - 1), PERSON_COLOURS[p % 4], 3)
        rows.append(tile)
    stacked = np.concatenate(rows, axis=0)
    return cv2.resize(stacked, (side, height)) if len(stacked) != height else stacked


def draw_person(image, p, i):
    outline(image, masks[p, i], PERSON_COLOURS[p % 4])
    points = keypoints[p, i]
    for a, b in COCO_BONES:
        if np.isfinite(points[[a, b]]).all():
            colour = confidence_colour(min(points[a, 2], points[b, 2]))
            cv2.line(image, points[a, :2].astype(int), points[b, :2].astype(int), colour, 2)
    for x, y, c in points[np.isfinite(points).all(1)]:
        cv2.circle(image, (int(x), int(y)), 4, confidence_colour(c), -1)
    extra = all_keypoints[p, i, 17:]  # feet, face, hands: small dots, the sure ones only
    for x, y, c in extra[np.isfinite(extra).all(1) & (extra[:, 2] >= 0.3)]:
        cv2.circle(image, (int(x), int(y)), 2, confidence_colour(c), -1)
    for hand in hands[p, i]:
        if np.isfinite(hand).all():
            for a, b in HAND_BONES:
                cv2.line(image, hand[a, :2].astype(int), hand[b, :2].astype(int), (255, 220, 0), 1)
            wrist = hand[0, :2].astype(int)
            cv2.putText(
                image, f"{hand[0, 2]:.2f}", wrist, cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 220, 0), 1
            )
    for x, y, _ in face_landmarks[p, i][np.isfinite(face_landmarks[p, i]).all(1)]:
        cv2.circle(image, (int(x), int(y)), 1, (0, 255, 255), -1)


def panels(i):
    frame = frames[i]
    detections = frame.copy()
    texts = []
    for p in range(PEOPLE):
        draw_person(detections, p, i)
        if usable[p, i]:
            texts.append(f"P{p} {np.nanmean(keypoints[p, i, :, 2]):.2f}")
        else:
            texts.append(f"P{p} skipped (SAM {presence[p, i]:.2f})")
    label(detections, f"#{frame_indices[i]} keypoints " + " ".join(texts))

    parts = frame.copy()
    for p in range(PEOPLE):
        parts = tint(parts, body_parts[p, i], PART_COLOURS)
        outline(parts, masks[p, i], PERSON_COLOURS[p % 4], 1)
    label(parts, "body parts (Sapiens2)")

    confidence = part_confidence[:, i].astype(np.float32).max(axis=0)
    heat = (plt.get_cmap("magma")(confidence)[..., :3] * 255).astype(np.uint8)
    for p in range(PEOPLE):
        outline(heat, masks[p, i], PERSON_COLOURS[p % 4], 1)
    anyone = masks[:, i].any(0)
    mean = confidence[anyone].mean() if anyone.any() else 0
    label(heat, f"confidence, mean on people {mean:.2f}")

    row = np.concatenate([detections, parts, head_rows(i, HEIGHT), heat], axis=1)
    width = round(row.shape[1] * PANEL_HEIGHT / row.shape[0])
    return cv2.resize(row, (width, PANEL_HEIGHT), interpolation=cv2.INTER_AREA)


video_path = out_dir / "overlay.mp4"
with timed("overlay video"):
    write_video(video_path, (panels(i) for i in range(COUNT)), fps)
print("written to", video_path)
display(Video(str(video_path), embed=True, width=1400))

The colours of the body parts and the face parts:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.4), gridspec_kw={"width_ratios": [3, 1]})
legends = (
    (axes[0], "body parts (Sapiens2)", PART_NAMES, PART_COLOURS, 6),
    (axes[1], "face parts (landmarks)", FACE_PART_NAMES, FACE_COLOURS, 2),
)
for ax, title, names, colours, columns in legends:
    for n, name in enumerate(names[1:], start=1):
        x, y = (n - 1) % columns, -((n - 1) // columns)
        ax.add_patch(plt.Rectangle((x, y), 0.95, 0.8, color=colours[n] / 255))
        ax.text(x + 0.47, y + 0.4, name, ha="center", va="center", fontsize=7, color="white")
    ax.set_xlim(0, columns)
    ax.set_ylim(-((len(names) - 2) // columns) - 0.2, 1)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## One frame, large

In [ ]:
def show_frame(i):
    row = panels(i)
    plt.figure(figsize=(22, 22 * row.shape[0] / row.shape[1]))
    plt.imshow(row)
    plt.axis("off")
    plt.show()
    for p in range(PEOPLE):
        pairs = zip(COCO_KEYPOINTS, keypoints[p, i, :, 2], strict=True)
        print(
            f"frame {frame_indices[i]}, person {p}: " + ", ".join(f"{n} {s:.2f}" for n, s in pairs)
        )


show_frame(select_frame)

## Seen in: each body part's chance of being reconstructed, per person

For every part of every person: in how many frames it shows (at least `MIN_PART_PIXELS`
pixels; `MIN_FEATURE_PIXELS` for the small face parts), from how many directions around them
(their yaw from shoulders and nose, in `YAW_BINS` bins; rough when they are not upright), how
large it shows (median pixels), and how sure the model is of it. The **reconstruction** score
multiplies four shares, each capped at 1:

- views: directions seen / `YAW_BINS` (a part seen only from the front has no back),
- frames: frames seen / `ENOUGH_FRAMES`,
- detail: median pixels / `ENOUGH_PIXELS` (`ENOUGH_FEATURE_PIXELS` for face parts),
  square-rooted (a texture needs pixels per side),
- certainty: Sapiens2's mean confidence on the part; for face parts and hands, the face's and
  the hands' scores.

It is a heuristic for where to expect trouble, not a measured error: a low score means that
part will come from the model's prior or its neighbours, not from what the video showed.

In [ ]:
def part_rows(p):
    # (name, frames seen, directions, median pixels, certainty, score) per part, best first
    yaw = body_yaw(keypoints[p], min_confidence=MIN_KEYPOINT)
    direction = np.floor(((yaw + np.pi / YAW_BINS) % (2 * np.pi)) / (2 * np.pi) * YAW_BINS)
    entries = []  # name, pixels per frame, certainty, whether a small face part
    for k, name in enumerate(PART_NAMES[1:], start=1):
        area = body_parts[p] == k
        if area.any():
            certainty = float(part_confidence[p][area].astype(np.float32).mean())
            entries.append((name, area.reshape(COUNT, -1).sum(1), certainty, False))
    found_face = np.isfinite(face_confidence[p])
    face_certainty = float(np.nanmean(face_confidence[p])) if found_face.any() else 0.0
    for k, name in enumerate(FACE_PART_NAMES[1:], start=1):
        counts = (face_parts[p] == k).reshape(COUNT, -1).sum(1)
        if counts.any():
            entries.append((f"face: {name}", counts, face_certainty, True))
    for side, name in enumerate(("left hand (landmarks)", "right hand (landmarks)")):
        found = np.isfinite(hands[p, :, side, 0, 0])
        if found.any():
            span = np.nanmax(hands[p, :, side, :, :2], 1) - np.nanmin(hands[p, :, side, :, :2], 1)
            counts = np.where(found, np.nan_to_num(span[:, 0] * span[:, 1]), 0)
            entries.append((name, counts, float(np.nanmean(hands[p, :, side, 0, 2])), False))
    rows = []
    for name, counts, certainty, small in entries:
        seen = counts >= (MIN_FEATURE_PIXELS if small else MIN_PART_PIXELS)
        views = len(np.unique(direction[seen & np.isfinite(direction)]))
        detail = float(np.median(counts[seen])) if seen.any() else 0.0
        enough = ENOUGH_FEATURE_PIXELS if small else ENOUGH_PIXELS
        shares = (
            views / YAW_BINS,
            min(1, seen.sum() / ENOUGH_FRAMES),
            min(1, np.sqrt(detail / enough)),
            certainty,
        )
        rows.append((name, int(seen.sum()), views, detail, certainty, float(np.prod(shares))))
    return sorted(rows, key=lambda row: -row[5])


tables = [part_rows(p) for p in range(PEOPLE)]
for p, rows in enumerate(tables):
    print(f"\nperson {p}")
    print(f"{'part':<26}{'frames':>8}{'views':>8}{'median px':>11}{'certainty':>11}{'score':>8}")
    for name, seen, views, detail, certainty, score in rows:
        print(
            f"{name:<26}{seen:>8}{views:>5}/{YAW_BINS:<2}{detail:>11.0f}{certainty:>11.2f}{score:>8.2f}"
        )

height = 0.3 * max(len(rows) for rows in tables) + 1
fig, axes = plt.subplots(1, PEOPLE, figsize=(7 * PEOPLE, height), squeeze=False)
for p, ax in enumerate(axes[0]):
    rows = tables[p][::-1]
    scores = [row[5] for row in rows]
    ax.barh([row[0] for row in rows], scores, color=plt.get_cmap("RdYlGn")(scores))
    ax.set_xlim(0, 1)
    ax.set_title(f"person {p}: reconstruction score")
plt.tight_layout()
plt.show()

## Confidence over time

Per person: the keypoints' confidence per frame, the hands' and the face's, how much of them
each of the larger body parts covers, and the mean confidences. Dips mark frames to look at with
`show_frame`; the least sure frames are listed at the end.

In [ ]:
def mean_on(confidence, areas):
    # (N,) mean confidence inside each frame's area; NaN where it is empty
    return [c[m].mean() if m.any() else np.nan for c, m in zip(confidence, areas, strict=True)]


for p in range(PEOPLE):
    fig, axes = plt.subplots(4, 1, figsize=(16, 14), gridspec_kw={"height_ratios": [2, 1, 1.4, 1]})
    fig.suptitle(f"person {p}")
    image = axes[0].imshow(
        keypoints[p, ..., 2].T,
        aspect="auto",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )
    axes[0].set_yticks(range(len(COCO_KEYPOINTS)), COCO_KEYPOINTS, fontsize=8)
    axes[0].set_title("body keypoint confidence")
    plt.colorbar(image, ax=axes[0], fraction=0.02)

    axes[1].plot(hands[p, :, 0, 0, 2], label="left hand")
    axes[1].plot(hands[p, :, 1, 0, 2], label="right hand")
    axes[1].plot(face_confidence[p], label="face")
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title("hands (MediaPipe's score) and face (its keypoints'); gaps: not found")
    axes[1].legend()

    theirs = masks[p].reshape(COUNT, -1).sum(1).clip(min=1)
    shares = {
        name: (body_parts[p] == PART[name]).reshape(COUNT, -1).sum(1) / theirs
        for name in PART_NAMES[1:]
    }
    for name in sorted(shares, key=lambda n: -shares[n].sum())[:8]:
        axes[2].plot(shares[name], label=name)
    axes[2].set_title("share of their pixels, the eight largest body parts")
    axes[2].legend(ncol=4, fontsize=8)

    axes[3].plot(presence[p], label="SAM: present")
    axes[3].plot(mask_certainty[p], label="SAM: mask certainty")
    skipped = np.flatnonzero(~usable[p])
    axes[3].plot(skipped, np.zeros(len(skipped)), "kx", label="skipped")
    axes[3].plot(mean_on(part_confidence[p].astype(np.float32), masks[p]), label="body parts")
    axes[3].plot(np.nanmean(keypoints[p, ..., 2], axis=1), label="keypoints (mean)")
    axes[3].set_ylim(0, 1.05)
    axes[3].set_title("mean confidence")
    axes[3].set_xlabel("prepared frame")
    axes[3].legend()
    plt.tight_layout()
    plt.show()

    means = np.nan_to_num(np.nanmean(keypoints[p, ..., 2], axis=1), nan=1.0)
    least_sure = np.argsort(means)[:3]
    print(f"person {p}, least sure frames:", ", ".join(f"{i} ({means[i]:.2f})" for i in least_sure))

## Where the time went

Wall-clock seconds per step, GPU work included, and per frame. Model downloads and first-use
setup count too on a first run; run the notebook twice for the steady numbers.

In [ ]:
total = sum(timings.values())
print(f"{'step':<24}{'seconds':>9}{'ms/frame':>10}{'share':>8}")
for step, seconds in sorted(timings.items(), key=lambda item: -item[1]):
    print(f"{step:<24}{seconds:>9.1f}{1000 * seconds / COUNT:>10.0f}{seconds / total:>8.0%}")
print(f"{'total':<24}{total:>9.1f}{1000 * total / COUNT:>10.0f}")

plt.figure(figsize=(10, 3))
steps = sorted(timings, key=timings.get)
plt.barh(steps, [timings[step] for step in steps])
plt.xlabel("seconds")
plt.title(f"preparing {COUNT} frames at {WIDTH}x{HEIGHT}, {PEOPLE} people")
plt.show()